# LSTM Forecasting for AquaSense

This notebook trains a multivariate LSTM that forecasts the next timestep for all processed water-quality features. The model is exported as a loadable Keras artifact together with its scaler and metadata so it can be reused by the serving layer later.

In [9]:
import io
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras

np.random.seed(42)
tf.keras.utils.set_random_seed(42)
tf.get_logger().setLevel('ERROR')

In [10]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        data_file = candidate / 'docs' / 'data' / 'processed' / 'burgas_final.csv'
        if data_file.exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
DATA_PATH = REPO_ROOT / 'docs' / 'data' / 'processed' / 'burgas_final.csv'
ARTIFACT_DIR = REPO_ROOT / 'ml_system' / 'models' / 'artifacts' / 'lstm_forecaster'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOOKBACK = 60
FEATURE_COLUMNS = [
    'sea_level_m',
    'temperature_C',
    'dissolved_o2',
    'primary_production',
    'salinity_psu',
    'nitrate',
    'phosphate',
]
TARGET_COLUMNS = FEATURE_COLUMNS.copy()

print(f'Repository root: {REPO_ROOT}')
print(f'Data file: {DATA_PATH}')
print(f'Export directory: {ARTIFACT_DIR}')

Repository root: c:\Users\Students\AquaSense
Data file: c:\Users\Students\AquaSense\docs\data\processed\burgas_final.csv
Export directory: c:\Users\Students\AquaSense\ml_system\models\artifacts\lstm_forecaster


In [11]:
df = pd.read_csv(DATA_PATH, parse_dates=['time']).sort_values('time').set_index('time')
df = df[FEATURE_COLUMNS].copy()
df = df.interpolate(method='time').ffill().bfill()

display(df.head())
print(f'Rows: {len(df):,}')
print(f'Features: {len(FEATURE_COLUMNS)}')

,sea_level_m,temperature_C,dissolved_o2,primary_production,salinity_psu,nitrate,phosphate
time,,,,,,,
2022-05-27,0.328140,19.374306,281.69223,26.157421,18.357775,2.53451,0.223134
2022-05-28,0.330184,19.374306,281.69223,26.157421,18.345295,2.53451,0.223134
2022-05-29,0.331792,19.374306,281.69223,26.157421,18.335102,2.53451,0.223134
2022-05-30,0.330618,19.374306,281.69223,26.157421,18.348910,2.53451,0.223134
2022-05-31,0.331508,19.374306,281.69223,26.157421,18.337750,2.53451,0.223134


Rows: 370
Features: 7


In [12]:
split_train = int(len(df) * 0.70)
split_val = int(len(df) * 0.85)

train_df = df.iloc[:split_train]
val_df = df.iloc[split_train - LOOKBACK:split_val]
test_df = df.iloc[split_val - LOOKBACK:]

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_df)
val_scaled = scaler.transform(val_df)
test_scaled = scaler.transform(test_df)

def build_sequences(values: np.ndarray, lookback: int):
    inputs = []
    targets = []
    for index in range(lookback, len(values)):
        inputs.append(values[index - lookback:index])
        targets.append(values[index])
    return np.asarray(inputs, dtype=np.float32), np.asarray(targets, dtype=np.float32)

X_train, y_train = build_sequences(train_scaled, LOOKBACK)
X_val, y_val = build_sequences(np.vstack([train_scaled[-LOOKBACK:], val_scaled]), LOOKBACK)
X_test, y_test = build_sequences(np.vstack([val_scaled[-LOOKBACK:], test_scaled]), LOOKBACK)

print(f'Train shape: {X_train.shape} -> {y_train.shape}')
print(f'Validation shape: {X_val.shape} -> {y_val.shape}')
print(f'Test shape: {X_test.shape} -> {y_test.shape}')

Train shape: (199, 60, 7) -> (199, 7)
Validation shape: (115, 60, 7) -> (115, 7)
Test shape: (116, 60, 7) -> (116, 7)


## LSTM Model

The network predicts the next timestep for all seven features from the previous 30 timesteps. This is the shape the serving layer can later roll forward recursively for longer horizons such as 2 days or 1 year.

In [13]:
model = keras.Sequential([
    keras.layers.Input(shape=(LOOKBACK, len(FEATURE_COLUMNS))),
    # First LSTM layer with high capacity and return_sequences=True to pass to next layer
    keras.layers.LSTM(128, return_sequences=True, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    keras.layers.Dropout(0.3),
    # Second LSTM layer
    keras.layers.LSTM(64, return_sequences=True, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    keras.layers.Dropout(0.3),
    # Third LSTM layer (final LSTM outputs single time step)
    keras.layers.LSTM(32, return_sequences=False, activation='relu'),
    keras.layers.Dropout(0.2),
    # Dense layers for final processing
    keras.layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(len(TARGET_COLUMNS)),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss='mse',
    metrics=['mae'],
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 60, 128)        │        69,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 60, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 7)              │           231 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 135,879 (530.78 KB)

 Trainable params: 135,879 (530.78 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, min_delta=1e-4),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.7, patience=8, min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True, verbose=0),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=16,
    callbacks=callbacks,
    verbose=1,
)

test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
y_pred = model.predict(X_test, verbose=0)

y_test_actual = scaler.inverse_transform(y_test)
y_pred_actual = scaler.inverse_transform(y_pred)
rmse_by_feature = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual, multioutput='raw_values'))
mae_by_feature = mean_absolute_error(y_test_actual, y_pred_actual, multioutput='raw_values')

evaluation = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'mae': mae_by_feature,
    'rmse': rmse_by_feature,
})

print(f'Test loss: {test_loss:.6f}')
print(f'Test MAE: {test_mae:.6f}')
evaluation

Epoch 1/150


12/13 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.3108 - mae: 0.4475

13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - loss: 0.2968 - mae: 0.4334 - val_loss: 0.3693 - val_mae: 0.4830 - learning_rate: 5.0000e-04
Epoch 2/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.2145 - mae: 0.3574

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.1894 - mae: 0.3341 - val_loss: 0.2665 - val_mae: 0.4126 - learning_rate: 5.0000e-04
Epoch 3/150
12/13 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.1419 - mae: 0.2899

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.1332 - mae: 0.2764 - val_loss: 0.2463 - val_mae: 0.4069 - learning_rate: 5.0000e-04
Epoch 4/150
12/13 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.1151 - mae: 0.2505

...

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 0.0347 - mae: 0.1239 - val_loss: 0.0731 - val_mae: 0.1841 - learning_rate: 5.0000e-04
Epoch 20/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0325 - mae: 0.1195 - val_loss: 0.0805 - val_mae: 0.1902 - learning_rate: 5.0000e-04
Epoch 21/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0311 - mae: 0.1166 - val_loss: 0.0848 - val_mae: 0.1936 - learning_rate: 5.0000e-04
Epoch 22/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.0290 - mae: 0.1106 - val_loss: 0.0779 - val_mae: 0.2035 - learning_rate: 5.0000e-04
Epoch 23/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.0296 - mae: 0.1126 - val_loss: 0.0791 - val_mae: 0.1905 - learning_rate: 5.0000e-04
Epoch 24/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0287 - mae: 0.1114 - val_loss: 0.0827 - val_mae: 0.1939 - learning_rate: 5.0000e-04
Epoch 25/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0270 - mae: 0.1072 - val_loss: 0.0748 - val_mae: 0.1871 - learning_r

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.0261 - mae: 0.1044 - val_loss: 0.0725 - val_mae: 0.1843 - learning_rate: 5.0000e-04
Epoch 28/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0262 - mae: 0.1041 - val_loss: 0.1008 - val_mae: 0.2113 - learning_rate: 5.0000e-04
Epoch 29/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0268 - mae: 0.1058 - val_loss: 0.0759 - val_mae: 0.1801 - learning_rate: 5.0000e-04
Epoch 30/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0244 - mae: 0.1020 - val_loss: 0.0740 - val_mae: 0.1952 - learning_rate: 5.0000e-04
Epoch 31/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0236 - mae: 0.0982

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.0239 - mae: 0.0996 - val_loss: 0.0699 - val_mae: 0.1726 - learning_rate: 5.0000e-04
Epoch 32/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0240 - mae: 0.1001 - val_loss: 0.0721 - val_mae: 0.1915 - learning_rate: 5.0000e-04
Epoch 33/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0214 - mae: 0.0924 - val_loss: 0.0763 - val_mae: 0.1823 - learning_rate: 5.0000e-04
Epoch 34/150
12/13 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0246 - mae: 0.1021

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.0254 - mae: 0.1040 - val_loss: 0.0675 - val_mae: 0.1762 - learning_rate: 5.0000e-04
Epoch 35/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0220 - mae: 0.0947 - val_loss: 0.0692 - val_mae: 0.1695 - learning_rate: 5.0000e-04
Epoch 36/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0219 - mae: 0.0944 - val_loss: 0.0736 - val_mae: 0.1819 - learning_rate: 5.0000e-04
Epoch 37/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0215 - mae: 0.0946 - val_loss: 0.0708 - val_mae: 0.1865 - learning_rate: 5.0000e-04
Epoch 38/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0213 - mae: 0.0931 - val_loss: 0.0712 - val_mae: 0.1734 - learning_rate: 5.0000e-04
Epoch 39/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0222 - mae: 0.0958 - val_loss: 0.0736 - val_mae: 0.1806 - learning_rate: 5.0000e-04
Epoch 40/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0209 - mae: 0.0914 - val_loss: 0.0920 - val_mae: 0.2030 - learning_r

,feature,mae,rmse
0,sea_level_m,0.023945,0.028027
1,temperature_C,2.101120,2.993475
2,dissolved_o2,6.938088,7.415865
3,primary_production,6.060806,6.418050
4,salinity_psu,0.122042,0.135955
5,nitrate,0.732658,0.811768
6,phosphate,0.050353,0.067537


In [15]:
def forecast_steps(model, recent_window: np.ndarray, steps: int):
    window = np.asarray(recent_window, dtype=np.float32).copy()
    predictions = []
    for _ in range(steps):
        next_step = model.predict(window[None, ...], verbose=0)[0]
        predictions.append(next_step)
        window = np.vstack([window[1:], next_step])
    return scaler.inverse_transform(np.asarray(predictions, dtype=np.float32))

summary_lines = []
model.summary(print_fn=summary_lines.append)
summary_text = '\n'.join(summary_lines)

artifact_paths = {
    'model': ARTIFACT_DIR / 'lstm_forecaster.keras',
    'scaler': ARTIFACT_DIR / 'lstm_forecaster_scaler.joblib',
    'features': ARTIFACT_DIR / 'lstm_forecaster_features.json',
    'config': ARTIFACT_DIR / 'lstm_forecaster_config.json',
    'summary': ARTIFACT_DIR / 'lstm_forecaster_summary.txt',
}

model.save(artifact_paths['model'])
joblib.dump(scaler, artifact_paths['scaler'])
artifact_paths['features'].write_text(json.dumps(FEATURE_COLUMNS, indent=2), encoding='utf-8')
artifact_paths['summary'].write_text(summary_text, encoding='utf-8')

feature_metrics = [
    {
        'feature': feature,
        'mae': float(mae),
        'rmse': float(rmse),
    }
    for feature, mae, rmse in zip(FEATURE_COLUMNS, mae_by_feature, rmse_by_feature)
]

config = {
    'model_name': 'lstm_forecaster',
    'lookback': LOOKBACK,
    'feature_columns': FEATURE_COLUMNS,
    'target_columns': TARGET_COLUMNS,
    'forecast_frequency': '1D',
    'train_rows': int(len(train_df)),
    'validation_rows': int(len(val_df)),
    'test_rows': int(len(test_df)),
    'test_loss': float(test_loss),
    'test_mae': float(test_mae),
    'feature_metrics': feature_metrics,
    'date_range': {
        'start': df.index.min().isoformat(),
        'end': df.index.max().isoformat(),
    },
}

artifact_paths['config'].write_text(json.dumps(config, indent=2), encoding='utf-8')

print(f"Saved model: {artifact_paths['model']}")
print(f"Saved scaler: {artifact_paths['scaler']}")
print(f"Saved metadata: {artifact_paths['config']}")
print(f"Saved summary: {artifact_paths['summary']}")

example_forecast = forecast_steps(model, X_test[-1], steps=7)
pd.DataFrame(example_forecast, columns=FEATURE_COLUMNS)

Saved model: c:\Users\Students\AquaSense\ml_system\models\artifacts\lstm_forecaster\lstm_forecaster.keras
Saved scaler: c:\Users\Students\AquaSense\ml_system\models\artifacts\lstm_forecaster\lstm_forecaster_scaler.joblib
Saved metadata: c:\Users\Students\AquaSense\ml_system\models\artifacts\lstm_forecaster\lstm_forecaster_config.json
Saved summary: c:\Users\Students\AquaSense\ml_system\models\artifacts\lstm_forecaster\lstm_forecaster_summary.txt


,sea_level_m,temperature_C,dissolved_o2,primary_production,salinity_psu,nitrate,phosphate
0,0.331638,11.019952,286.041412,9.704881,18.493515,1.426483,0.257190
1,0.331683,11.120410,285.198914,9.892720,18.491989,1.418695,0.255295
2,0.331644,11.226549,284.368439,10.136033,18.490738,1.409110,0.253282
3,0.331470,11.335165,283.555115,10.434421,18.490021,1.398842,0.251106
4,0.331159,11.412525,282.897705,10.669410,18.489777,1.391276,0.248930
5,0.330737,11.459237,282.407715,10.844203,18.490044,1.386039,0.246835
6,0.330246,11.486308,282.068573,10.969186,18.490786,1.382231,0.244886


## Next Integration Step

The exported files in `ml_system/models/artifacts/lstm_forecaster/` are the inputs for the future forecast endpoint and web button. That endpoint can load the scaler and Keras model, accept a horizon such as `2d` or `1y`, and recursively roll the predictions forward.